In [15]:
from pathlib import Path
import numpy as np
import pandas as pd
import nd2
from ipywidgets import interact, IntSlider, widgets
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interactive_output
import tifffile as tiff
from scipy import ndimage
from nd2reader import ND2Reader
import nd2reader
from cellpose import models, io
from skimage.restoration import rolling_ball
from skimage.morphology import (
    disk,
    dilation,
    closing,
    remove_small_objects,
    remove_small_holes,
)

In [2]:
dark_400 = tiff.imread("/Volumes/TAYLOR-LAB/Huyen Anh /20221216 3nM 231-A8_NEMO_TRAF6_1um_38mol 001/20210531 Darkfield 400ms.tif").astype(np.float32)
dark_60 = tiff.imread("/Volumes/TAYLOR-LAB/Huyen Anh /20221216 3nM 231-A8_NEMO_TRAF6_1um_38mol 001/20210531 Darkfield 60ms.tif").astype(np.float32)

print(dark_400.shape)
print(dark_60.dtype)

(600, 600)
float32


In [4]:
og_file = "/Volumes/TAYLOR-LAB/Huyen Anh /20251210 3nM 540-D12_NEMO_TRAF6_HOIP-KO 1um_27mol/20251210 3nM 540-D12_NEMO_TRAF6_HOIP-KO 1um_27mol.nd2"
my_array = nd2.imread(og_file)
my_file = nd2.ND2File(og_file)
my_file.shape

channel_names = ["Red", "GFP", "BF"]

def show_frame(t):

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    #[Red]
    axes[0].imshow(
        my_array[t, 0],
        cmap="magma",
        vmin=0,
        vmax=500
    )
    axes[0].set_title("mScarlet")
    axes[0].axis("off")

    #[GFP]
    axes[1].imshow(
        my_array[t, 1],
        cmap="viridis",
        vmin=0,
        vmax=200
    )
    axes[1].set_title("GFP")
    axes[1].axis("off")

    #[Brightfield]
    axes[2].imshow(
        my_array[t, 2],
        cmap="gray"
        
    )
    axes[2].set_title("BF")
    axes[2].axis("off")

    plt.suptitle(f"Timepoint {t}")
    plt.tight_layout()
    plt.show()

play = widgets.Play(
    value=0,
    min=0,
    max=my_array.shape[0]-1,
    step=1,
    interval=3,      # ms/frame
)
slider = widgets.IntSlider(
    min=0,
    max=my_array.shape[0]-1,
    step=1,
    value=300,
    description="Time"
)

# Link play button to slider
widgets.jslink((play, 'value'), (slider, 'value'))

# Display
ui = widgets.HBox([play, slider])
out = interactive_output(show_frame, {'t': slider})

display(ui, out)

Output()

In [16]:
rfp_raw = my_array[:,0].astype(np.float32)
gfp_raw = my_array[:,1].astype(np.float32)
bf_raw = my_array[:,2].astype(np.float32)

# np.clip() -> I >= 0, makes sure that every pixel is equal or larger than zero
# Dark-frame subtraction
rfp_minusdark = np.clip(rfp_raw - dark_400, 0, None)
gfp_minusdark = np.clip(gfp_raw - dark_60, 0, None)
bf_minusdark  = np.clip(bf_raw - dark_60, 0, None)

def show_Imageminusdarkframe(t):

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # RFP
    axes[0].imshow(
        rfp_minusdark[t],
        cmap="magma",
        vmin=0,
        vmax=500
    )
    axes[0].set_title("mScarlet (dark-frame corrected)")
    axes[0].axis("off")

    # GFP
    axes[1].imshow(
        gfp_minusdark[t],
        cmap="viridis",
        vmin=0,
        vmax=200
    )
    axes[1].set_title("GFP (dark-frame corrected)")
    axes[1].axis("off")

    # Brightfield
    axes[2].imshow(
        bf_minusdark[t],
        cmap="gray"
    )
    axes[2].set_title("Brightfield")
    axes[2].axis("off")

    plt.suptitle(f"Timepoint {t}")
    plt.tight_layout()
    plt.show()

play = widgets.Play(
    value=0,
    min=0,
    max=rfp_minusdark.shape[0]-1,
    step=1,
    interval=3      # ms/frame
)

slider = widgets.IntSlider(
    value=300,
    min=0,
    max=rfp_minusdark.shape[0]-1,
    step=1,
    description="Time"
)

widgets.jslink((play, "value"), (slider, "value"))

ui = widgets.HBox([play, slider])
out = interactive_output(show_Imageminusdarkframe, {"t": slider})

display(ui, out)

Output()

# rolling Ball Algorithm
The rolling-ball algorithm estimates the background intensity of a grayscale image. It comes in useful, for instance, in case of uneven exposure, when subtracting the background is desirable. It is frequently used in biomedical image processing and was first proposed by Stanley R. Sternberg in 1983 [1].\
The algorithm works as a filter: Think of the image as a surface that has unit-sized blocks stacked on top of each other in place of each pixel. The number of blocks, and hence surface height, is determined by the intensity of the pixel. To get the intensity of the background at a desired (pixel) position, we imagine submerging a ball under the surface at the desired position. Once it is completely covered by the blocks, the apex of the ball determines the intensity of the background at that position. We can then ‘roll’ this ball around below the surface to get the background values for the entire image. The larger the ball, the smoother the background.\
\
scikit-image implements a generalized version of this rolling-ball algorithm, allowing you to work with n-dimensional images and to use not only balls, but other kernels as well. This way, you may directly filter RGB images or image stacks along any (or all) spatial dimensions.\

In [19]:
def roll_ball_algorithm(movie, radius:int):
    """using the rolling ball algorithm to smooth out the background without disturbing the actual signal

    Args:
        movie (array): (time, X, Y) shape array (dark framed corrected)
        radius (int): radius of the ball, the bigger the smoother

    Returns:
        Array: movie with smooth out background
    """
    
    result = np.empty_like(movie)

    for t in range(movie.shape[0]):

        background = rolling_ball(movie[t], radius= radius)

        result[t] = np.clip(movie[t]- background, 0, None)

    return result

def gaussian_filter(movie, sigma):
    """appling the gaussian filter, to spread the fluorescence slightly (dont use huge numbers, sigma = 0.5 is actually good enough)

    Args:
        movie (array): (time, X, Y) shape array (dark framed corrected and can be smoothed out already)
                radius (int): radius of the ball, the bigger the smoother

    Returns:
        Array: movie with Gaussian Blur
    """
    
    result = np.empty_like(movie)

    for t in range(movie.shape[0]):

        result[t] = ndimage.gaussian_filter(movie[t], sigma= sigma)

    return result

def contrastenhance(movie, min_percentile, max_percentile):
    """stretching the contrast between the choosen percentile. (percentile (20,98) is good if background should stay dark and the puncta kina bright)

    Args:
        movie (Array): (time, X, Y) shape array (dark framed corrected and can be smoothed out already)
        min_percentile (float or int): minimal percentile
        max_percentile (float or int): maximal percentile

    Returns:
        Array: movie with stretched contrast
    """
    
    result = np.empty_like(movie)

    for t in range(movie.shape[0]):
        pmin, pmax = np.percentile(movie[t], (min_percentile, max_percentile))
        result[t] = rescale_intensity(
                                    movie[t],
                                    in_range=(pmin, pmax)
                                    )

    return result


def mask(movie, cutoff_percentile, punctasize = 5, disk_expand = 6, disk_closing = 10, area_threshold = 1000, removsize = 200):
    """Masking the movie. Best is to use a high percentile to make sure the backgroundcells are also excluded

    Args:
        movie (array): _description_
        cutoff_percentile (float): np.percentile(image, percentile)
        punctasize (int, optional): punctasize. Defaults to 5.
        disk_expand (int, optional): enlarge bright region and shrink dark region (footprintsize). Defaults to 6.
        disk_closing (int, optional): close up gaps (footprintsize). Defaults to 10.
        area_threshold (int, optional): Remove contiguous holes smaller than the specified size. Defaults to 1000.
        removsize (int, optional): Remove objects smaller than the specified size. Defaults to 200.

    Returns:
        _type_: _description_
    """
    result = np.empty_like(movie)

    for i in range(movie.shape[0]):
        image = movie[i]
        threshold = np.percentile(image, cutoff_percentile)
        puncta = image >= threshold
        # result[i] = (np.where(movie[i] < np.percentile(movie[i], cutoff_percentile), 0, movie[i]))
        # Remove isolated noise
        puncta = remove_small_objects(
            puncta,
            min_size= punctasize
        )

        expanded = dilation(
            puncta,
            footprint=disk(disk_expand)
        )

        expanded = closing(
            expanded,
            footprint=disk(disk_closing)
        )

        mask = remove_small_holes(
            expanded,
            area_threshold=area_threshold
        )

        mask = remove_small_objects(
            mask,
            min_size=removsize
        )
        result[i] = mask

    return result

def save_movie_for_cellpose(movie, output_folder, foldername : str, every_n=4):
    """
    Save every nth frame of a movie as individual TIFF files.

    Parameters
    ----------
    movie : ndarray
        Shape = (time, y, x)
    output_folder : str or Path
    foldername : str
    every_n : int
        Save every nth frame.
    """

    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    saved = 0

    for frame in range(0, movie.shape[0], every_n):

        image = movie[frame].astype(np.float32)

        filename = output_folder /  f"{foldername}_{frame:04d}.tif"

        tiff.imwrite(filename, image)

        saved += 1

    print(f"{output_folder.name}: {saved} images saved.")


In [20]:
# [applying smoothing algorithm]
# RFP
rfp_roolingball = roll_ball_algorithm(rfp_minusdark, 50)
rfp_gaussian = gaussian_filter(rfp_roolingball, 0.5)
# rfp_conenh = contrastenhance(rfp_roolingball, 20, 98)
# rfp_gaus_conenh = contrastenhance(rfp_gaussian, 20, 98)

# GFP
gfp_roolingball = roll_ball_algorithm(gfp_minusdark, 50)
gfp_gaussian = gaussian_filter(gfp_roolingball, 0.5)
# gfp_conenh = contrastenhance(gfp_roolingball, 20, 98)
# gfp_gaus_conenh = contrastenhance(gfp_gaussian, 20, 98)

In [21]:
base = Path("cellpose_training/20251210 3nM 540-D12_NEMO_TRAF6_HOIP-KO 1um_27mol")

save_movie_for_cellpose(rfp_roolingball,   base / "rfp_rollingball", "rfp_rollingball")
save_movie_for_cellpose(rfp_gaussian,      base / "rfp_gaussian", "rfp_gaussian")

save_movie_for_cellpose(gfp_roolingball,   base / "gfp_rollingball", "gfp_rollingball")
save_movie_for_cellpose(gfp_gaussian,      base / "gfp_gaussian", "gfp_gaussian")


rfp_rollingball: 113 images saved.
rfp_gaussian: 113 images saved.
gfp_rollingball: 113 images saved.
gfp_gaussian: 113 images saved.


In [11]:
from pathlib import Path
import zipfile

import numpy as np
import tifffile
from roifile import ImagejRoi
from skimage.draw import polygon


roi_zip = Path(
    "/Users/huyenanh/git_repos/MT_IRAK4/cellpose_training/20251210 3nM 540-D12_NEMO_TRAF6_HOIP-KO 1um_27mol/rfp_rollingball/rfp_rollingball_masks_0120.zip"
)


output_mask = Path(
    "/Users/huyenanh/git_repos/MT_IRAK4/rfp_rollingball_masks_0120.tif"
)

height = 600
width = 600

# Start with background
mask = np.zeros((height, width), dtype=np.uint16)


with zipfile.ZipFile(roi_zip) as z:

    roi_names = [
        name for name in z.namelist()
        if name.lower().endswith(".roi")
    ]

    print(f"Found {len(roi_names)} ROIs")

    for label, roi_name in enumerate(roi_names, start=1):

        with z.open(roi_name) as f:
            roi = ImagejRoi.frombytes(f.read())

        coords = roi.coordinates()

        if coords is None:
            print(f"Skipping {roi_name}")
            continue

        # coordinates are x, y
        x = coords[:, 0]
        y = coords[:, 1]

        # Convert polygon into pixels
        rr, cc = polygon(y, x, shape=mask.shape)

        # Assign unique cell label
        mask[rr, cc] = label


tifffile.imwrite(output_mask, mask)

print("Saved:", output_mask)
print("Labels:", np.unique(mask))

Found 30 ROIs
Saved: /Users/huyenanh/git_repos/MT_IRAK4/rfp_rollingball_masks_0120.tif
Labels: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30]
